In [7]:
import lsdstreamburn.lsdstreamburn as sb

In [ ]:
my_dem = sb.get_dem(OT_api_key_fname="my_OT_api_key.txt", 
                    source="SRTM30", 
                    lower_left=[25.42, 77.48], 
                    upper_right = [31.48, 82.08],
                    prefix = "Chamoli_test")

print(my_dem)

In [ ]:
# Getting basin outline using Hydroshed basin boundary from running GEE script

import geopandas as gpd
import pandas as pd
import json
from shapely.geometry import shape
import rasterio
from rasterio.plot import show
import matplotlib.pyplot as plt

# Paths
csv_path = '../Chamoli_basin_outline_hydroshed.csv'
dem_path = 'Chamoli_SRTMGL1.tif'
output_plot = 'basin_outline_over_dem.png'

# Load basin outline
df = pd.read_csv(csv_path)
df['geometry'] = df['.geo'].apply(lambda x: shape(json.loads(x)))
gdf = gpd.GeoDataFrame(df, geometry='geometry', crs='EPSG:4326')

# Open DEM
with rasterio.open(dem_path) as src:
    fig, ax = plt.subplots(figsize=(10, 10))
    show(src, ax=ax, title="Basin outline over DEM", cmap='terrain')

    # Reproject if needed
    if gdf.crs != src.crs:
        gdf = gdf.to_crs(src.crs)

    gdf.boundary.plot(ax=ax, edgecolor='black', linewidth=2)

    plt.tight_layout()
    plt.savefig(output_plot, dpi=300)


In [ ]:
# Getting water mask on GEE Python API
import ee

import sys
sys.path.append('../geeCenterline')
import geeCenterline as geec # Use this package to dilate water mask and remove noise and holes

ee.Authenticate()
ee.Initialize()

from shapely.geometry import mapping
geom = df['geometry'].iloc[0]  # shapely.geometry.Polygon
geom_geojson = mapping(geom)  # gives a dict with 'type' and 'coordinates'

aoi = ee.Geometry.Polygon(geom_geojson['coordinates'])
start_date = ee.Date("2021-01-01")
end_date = start_date.advance(2, 'month')

# Apply filters to select relevant images
colFilter = ee.Filter.And(
    ee.Filter.bounds(aoi),
    ee.Filter.date(start_date, end_date)
)
DW_collection = ee.ImageCollection('GOOGLE/DYNAMICWORLD/V1').filter(colFilter)
print("Available Images:", DW_collection.size().getInfo())

# Select only the 'label' band
labelCol = DW_collection.select('label')

# 1) Create a binary water mask (1 if water, 0 otherwise)
def binary_water_mask(img):
    return img.eq(0)  # Water is 1, non-water is 0

waterMasks = labelCol.map(binary_water_mask)

# 2) Sum of water masks to count how many times a pixel is water
countWater = waterMasks.sum()

# 3) Count of valid pixels (not masked)
countValid = labelCol.count()

# 4) Compute water fraction per pixel
waterFraction = countWater.divide(countValid).reproject(
            crs=labelCol.first().projection(), 
            scale=10  # Explicitly setting scale to match Dynamic world scale 
        )

# 5) Threshold > 0.75 (75% water presence)
waterMajority = waterFraction.gt(0.75).rename('majorityWater').reproject(
            crs=labelCol.first().projection(), 
            scale=10
        )

# Clip to AOI
waterMajorityClipped = waterMajority.clip(aoi)

# connect the water masks divided by small gaps. The radius is the half scale of the gap to be filled.
waterLS1 = geec.close(waterMajorityClipped, radius=1.5, kernelType='square')

# Obtain the river mask
# identify the river from other water masks. The second input is the minimum area that is considered as the river.
# The second input is the connectivity of pixels. 4 means 4-connectivity and 8 means 8-connectivity.
riverMask = waterMajorityClipped.gt(0)
DWwater = waterMajorityClipped.eq(1).updateMask(riverMask)
DWwater2 = geec.close(DWwater, radius=1.5)
riverPS = geec.noise_removal(DWwater2, 500, 8)

Available Images: 242


In [ ]:
# Download a big channel mask in batch using geemap tiling
import geemap

# Create a fishnet grid over the AOI
fishnet = geemap.fishnet(aoi, h_interval=2.0, v_interval=2.0, delta=1)

# Download tiles in parallel
geemap.download_ee_image_tiles_parallel(
    image=riverPS,
    features=fishnet,
    out_dir="water_mask",       # Output directory (relative or absolute path)
    scale=10,                   # Spatial resolution in meters
    crs="EPSG:4326",            # Output projection
    num_threads=2               # 1 or 2 to avoid data quota limitation on GEE
)

In [ ]:
# Merging tiles of water mask in fishnet
import rasterio
from rasterio.merge import merge
from pathlib import Path

# List all .tif tiles
tile_dir = Path("water_mask")
tif_files = list(tile_dir.glob("*.tif"))

# Open all files
src_files_to_mosaic = [rasterio.open(str(fp)) for fp in tif_files]

# Merge into a single mosaic
mosaic, out_transform = merge(src_files_to_mosaic)

# Use metadata from first tile
out_meta = src_files_to_mosaic[0].meta.copy()
out_meta.update({
    "height": mosaic.shape[1],
    "width": mosaic.shape[2],
    "transform": out_transform,
    "driver": "GTiff"
})

# Save combined output
with rasterio.open("DW_mask.tif", "w", **out_meta) as dest:
    dest.write(mosaic)

# Close all open files
for src in src_files_to_mosaic:
    src.close()

In [ ]:
# Apply stream burn algorithm and extract channel network

# Define burn depths of water and sediment pixels
WATER_DEPTH = 30
SEDI_DEPTH = 0

# Use the DEM file generated from last cell
DEM_FNAME = 'Chamoli_SRTMGL1.tif'

# This channel mask comes from land cover classification from Dynamic World land cover product
CHANNEL_MASK_FNAME = 'DW_mask.tif' 

# Define a prefix for the file name of generated river network
LOCATION_YEAR = 'Chamoli2021'  

burned_dem_path = sb.burning_driver(DataDirectory = "./", 
                   dem_fname=DEM_FNAME, 
                   channel_mask_fname=CHANNEL_MASK_FNAME,
                   location_year = LOCATION_YEAR, 
                   burn_water_depth=WATER_DEPTH,
                   burn_sediment_depth=SEDI_DEPTH,
                   area_thresh=15000,
                   resolution=10,
                   dem_source='SRTM30')

print(burned_dem_path)

In [ ]:
burned_dem_prefix = burned_dem_path.split('/')[-1].split('.')[0] + '_UTM'

sb.plot_network(DataDirectory="./burned_dem/", DEM_prefix=burned_dem_prefix)